# Build a Real-Time Voice Assistant with Mistral and Agora

**Author:** Jingyi (@bluemotional)  
**Category:** voice, agent, real-time  
**Party:** Agora

This notebook shows how to build a real-time voice assistant with **Mistral as the LLM brain** and **Agora as the real-time voice runtime**.

For the fastest path, use Agora's official `recipe-agent-custom-llm` project, replace its mock LLM endpoint with Mistral, and run the live agent in the browser.


## 1. What you are building

You are building a browser-based voice assistant where Mistral generates the replies and Agora handles the live voice loop.


## 2. Fastest path

1. Install the Agora CLI and log in

```bash
curl -fsSL https://raw.githubusercontent.com/AgoraIO/cli/main/install.sh | sh
agora login
```

2. Create a new Agora project for this demo and verify it

```bash
agora project create mistral-voice --feature rtc --feature rtm --feature convoai
agora project use mistral-voice
agora project doctor --json
```

3. Clone the recipe and install dependencies

```bash
git clone https://github.com/AgoraIO-Conversational-AI/recipe-agent-custom-llm
cd recipe-agent-custom-llm
bun run setup
agora project env write server/.env.local
```

4. Copy the code block from the next section into `server/src/llm.py`.

5. Add these values to `server/.env.local`:

```bash
CUSTOM_LLM_MODEL=mistral-small-latest
CUSTOM_LLM_API_KEY=<your-mistral-key>
MISTRAL_SYSTEM_PROMPT=You are a concise, friendly real-time voice assistant powered by Mistral. Keep most replies to one or two short sentences unless the user asks for more detail.
AGENT_GREETING=Hi there! I'm your voice assistant. How can I help?
CUSTOM_LLM_URL=https://<your-tunnel>/llm/chat/completions
```

6. In a separate terminal window or tab, start a tunnel

```bash
ngrok http 8000
# or: cloudflared tunnel --url http://localhost:8000
```

Then go back to `server/.env.local` and replace `<your-tunnel>` in `CUSTOM_LLM_URL` with the public tunnel address.


## 3. Replace the mock LLM with Mistral

Copy the entire code block below into `server/src/llm.py`.


In [ ]:
import json
import os
import time
import uuid
from typing import Optional

import httpx
import uvicorn
from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse

MISTRAL_URL = os.getenv("MISTRAL_FORWARD_URL", "https://api.mistral.ai/v1/chat/completions")
DEFAULT_MODEL = os.getenv("CUSTOM_LLM_MODEL", "mistral-small-latest")
DEFAULT_SYSTEM_PROMPT = os.getenv(
    "MISTRAL_SYSTEM_PROMPT",
    "You are a concise, friendly real-time voice assistant powered by Mistral. "
    "Keep most replies to one or two short sentences unless the user asks for more detail. "
    "Speak naturally; do not use markdown or bullet points.",
)

app = FastAPI(title="Mistral adapter for Agora Conversational AI")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

def _normalize_messages(messages):
    out = []
    for m in messages or []:
        content = m.get("content")
        if isinstance(content, list):
            content = "".join(
                p.get("text", "") if isinstance(p, dict) else str(p) for p in content
            )
        out.append({"role": m.get("role"), "content": content})
    return out

def _ensure_system_message(messages):
    if messages and messages[0].get("role") == "system":
        return messages
    return [{"role": "system", "content": DEFAULT_SYSTEM_PROMPT}, *messages]

def _clean_for_mistral(body):
    payload = {
        "model": body.get("model") or DEFAULT_MODEL,
        "messages": _ensure_system_message(_normalize_messages(body.get("messages"))),
        "stream": True,
    }
    for k in ("temperature", "max_tokens", "top_p"):
        if body.get(k) is not None:
            payload[k] = body[k]
    return payload

def _make_role_chunk(chunk_id, model):
    chunk = {
        "id": chunk_id,
        "object": "chat.completion.chunk",
        "created": int(time.time()),
        "model": model,
        "choices": [{"index": 0, "delta": {"role": "assistant", "content": ""}, "finish_reason": None}],
    }
    return f"data: {json.dumps(chunk)}\n\n".encode()

def _make_stop_chunk(chunk_id, model):
    chunk = {
        "id": chunk_id,
        "object": "chat.completion.chunk",
        "created": int(time.time()),
        "model": model,
        "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}],
    }
    return f"data: {json.dumps(chunk)}\n\n".encode()

def _extract_delta(line):
    if not line.startswith("data:"):
        return None
    data = line[5:].strip()
    if not data or data == "[DONE]":
        return data
    payload = json.loads(data)
    choices = payload.get("choices") or []
    if not choices:
        return None
    return payload

@app.post("/chat/completions")
async def chat_completions(request: Request, authorization: Optional[str] = Header(None)):
    body = await request.json()
    if body.get("stream") is False:
        raise HTTPException(status_code=400, detail="Only streaming mode is supported. Set stream=true.")

    payload = _clean_for_mistral(body)
    headers = {"Authorization": authorization or "", "Content-Type": "application/json"}

    async def gen():
        chunk_id = f"chatcmpl-{uuid.uuid4().hex[:12]}"
        model = payload["model"]
        role_sent = False
        stop_sent = False

        async with httpx.AsyncClient(timeout=60) as cx:
            async with cx.stream("POST", MISTRAL_URL, headers=headers, json=payload) as r:
                if r.status_code != 200:
                    if not role_sent:
                        yield _make_role_chunk(chunk_id, model)
                        role_sent = True
                    if not stop_sent:
                        yield _make_stop_chunk(chunk_id, model)
                        stop_sent = True
                    yield b"data: [DONE]\n\n"
                    return

                async for line in r.aiter_lines():
                    if not line:
                        continue

                    parsed = _extract_delta(line)
                    if parsed is None:
                        continue

                    if parsed == "[DONE]":
                        if not role_sent:
                            yield _make_role_chunk(chunk_id, model)
                            role_sent = True
                        if not stop_sent:
                            yield _make_stop_chunk(chunk_id, model)
                            stop_sent = True
                        yield b"data: [DONE]\n\n"
                        return

                    choices = parsed.get("choices") or []
                    delta = choices[0].get("delta") or {}

                    if not role_sent:
                        yield _make_role_chunk(chunk_id, model)
                        role_sent = True

                    content = delta.get("content")
                    finish_reason = choices[0].get("finish_reason")
                    if content:
                        chunk = {
                            "id": chunk_id,
                            "object": "chat.completion.chunk",
                            "created": int(time.time()),
                            "model": model,
                            "choices": [{"index": 0, "delta": {"content": content}, "finish_reason": None}],
                        }
                        yield f"data: {json.dumps(chunk)}\n\n".encode()

                    if finish_reason and not stop_sent:
                        yield _make_stop_chunk(chunk_id, model)
                        stop_sent = True

                if not role_sent:
                    yield _make_role_chunk(chunk_id, model)
                if not stop_sent:
                    yield _make_stop_chunk(chunk_id, model)
                yield b"data: [DONE]\n\n"

    return StreamingResponse(gen(), media_type="text/event-stream")

@app.get("/health")
async def health():
    return {"status": "ok"}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=int(os.getenv("CUSTOM_LLM_PORT", "8001")))


## 4. Run the live voice agent

In your main terminal, run the app. Then open the printed web URL and click **Start Conversation**.

```bash
bun run dev
```


## Optional notes

- The adapter exists because direct forwarding can fail when Agora includes provider-specific metadata in the request body.
- The adapter also returns an OpenAI-style stream so the recipe and client can consume it reliably.


## Troubleshooting

- **Conversation fails to start**: run `agora project doctor --json` again and make sure the new project is ready for Conversational AI.
- **Tunnel URL stopped working**: restart the tunnel and update `CUSTOM_LLM_URL`.
- **No microphone audio**: grant browser microphone permission.
